## Exercise 5: Geospatial wrangling and making maps

Skills: 
* More geospatial practice building on earlier skills
* Make a map with `folium` or `ipyleaflet` using `shared_utils.map_utils`

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [1]:
import geopandas as gpd
import intake
import pandas as pd

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

from shapely.geometry import Polygon, LineString, Point 

import shared_utils
import altair as alt
from shared_utils import altair_utils 

import branca
# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command


/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.9.1-CAPI-1.14.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


## Research Question: What's the average number of trips per stop by operators in southern California? Show visualizations at the operator and county-level.
<br>**Geographic scope:** southern California counties
<br>**Deliverables:** chart(s) and map(s) showing metrics comparing across counties and also across operators. Make these visualizations using function(s).

### Prep data

* Use the same query, but grab a different set of operators. These are in southern California, so the map should zoom in counties ranging from LA to SD.
* To find what ITP IDs are what operators:
[agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* *Hint*: for some counties, there are multiple operators. Make sure the average trips per stop by counties is the weighted average.
* Use the same [shapefile for CA counties](https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12) as in Exercise 4.
* Join the data and only keep counties that have bus stops.


In [2]:
#choosing a different set of ITP IDS
ITP_ID = [182, 183, 278, 154, 235, 232]

In [3]:
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

/opt/conda/lib/python3.9/site-packages/siuba/sql/utils.py:52: SAWarning: Dialect bigquery:bigquery will not make use of SQL compilation caching as it does not set the 'supports_statement_cache' attribute to ``True``.  This can have significant performance implications including some performance degradations in comparison to prior SQLAlchemy versions.  Dialect maintainers should seek to set this attribute to True after appropriate development and testing for SQLAlchemy 1.4 caching support.   Alternatively, this attribute may be set to False which will disable this warning. (Background on this error at: https://sqlalche.me/e/14/cprf)


In [4]:
#creating point geometry
stops = shared_utils.geography_utils.create_point_geometry(stops, 'stop_lon', 'stop_lat')

In [5]:
#bringing in CA dataframe
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson').to_crs(epsg=4326)

In [6]:
#converting crs 
stops = stops.to_crs(epsg=4326)

In [7]:
geojson.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [8]:
#Joining the 2 dataframes, only keeping stops that are in a county
stops_joined = gpd.sjoin(stops, geojson, how='inner')

### Bring in a new table from BigQuery

* In `transitstacks`, there's a table called `ntd_stats`. 
* Decide what columns to keep.
* Merge `ntd_stats` with the `stops` table to create 1 df.

In [9]:
ntd_stats = (tbl.transitstacks.ntd_stats()
             >> select(_.transit_provider, _.itp_id, _.modes, _.upt_total_2019)
             >> collect()
             >> filter(_.itp_id.isin(ITP_ID))
            )

In [10]:
ntd_stats

,transit_provider,itp_id,modes,upt_total_2019
189,Laguna Beach Municipal Transit,154,MB,"820,829"
191,Los Angeles Metro,182,"HR,RB,LR,MB,MB,VP","379,718,121"
375,Orange County Transportation Authority,235,"DT,VP,DR,MB,CB,MB,CB","40,743,654"
380,Los Angeles Department of Transportation,183,"CB,DT,DR,MB","19,292,677"
387,OmniTrans,232,"DR,MB,MB","10,863,530"
391,San Diego Metropolitan Transit System,278,"MB,MB,CB,DT,DR,LR","85,357,495"


### Merging the 2
* Make sure all stop ids are unique

In [11]:
df2 = stops_joined.merge(ntd_stats, how = 'left', on ='itp_id')

In [12]:
df2.shape

(39310, 20)

In [41]:
f'out of the 39,310 rows, only {df2.stop_id.nunique()} are unique'

'out of the 39,310 rows, only 22356 are unique'

In [42]:
#new dataframe with only unique stop ids
df3 = df2.drop_duplicates(subset=['stop_id'])

In [43]:
#checking that the rows make sense.
df3.shape

(22356, 20)

In [44]:
df3.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,transit_provider,modes,upt_total_2019
0,154,19113,33.544868,-117.782707,Laguna Beach Bus Depot,POINT (-117.78271 33.54487),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.15861,0.20129,Laguna Beach Municipal Transit,MB,"820,829"
1,154,19114,33.540688,-117.780060,Legion Hall,POINT (-117.78006 33.54069),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.15861,0.20129,Laguna Beach Municipal Transit,MB,"820,829"


### Aggregate 
<b> Instructions </b> 
* Write a function to aggregate to the operator level or county level, add new columns for desired metrics.
* Merge in CA shapefile to get a gdf.
* Add another `geometry` column, called `centroid`, and grab the county's centroid.
* Refer to [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.set_geometry.html) to see how to pick which column to use as the `geometry` for the gdf, since technically, a gdf can handle multiple geometry columns.



In [52]:
#subset for count of stop ids by county.
subset1 = df3[['COUNTY_NAME','stop_id','geometry']]

In [58]:
#subset for count passengers.
subset2 = df3[['COUNTY_NAME','upt_total_2019','geometry']]

In [54]:
#function to aggregate to the operator level or county level, add new columns for desired metrics.
def aggregation_nunique(df, aggregated_column):
    df = df.dissolve(by=aggregated_column, aggfunc ='nunique').reset_index()
    return df 

In [56]:
def aggregation_sum(df, aggregated_column):
    df = df.dissolve(by=aggregated_column, aggfunc ='sum').reset_index()
    return df 

In [55]:
county_stops = aggregation_nunique(subset1, 'COUNTY_NAME')
county_stops

,COUNTY_NAME,geometry,stop_id
0,Los Angeles,"MULTIPOINT (-118.84339 34.03152, -118.82113 34...",13181
1,Orange,"MULTIPOINT (-118.10607 33.74889, -118.10577 33...",5579
2,Riverside,"MULTIPOINT (-117.55461 33.99433, -117.46784 33...",2
3,San Bernardino,"MULTIPOINT (-117.73260 34.00136, -117.72986 34...",337
4,San Diego,"MULTIPOINT (-117.27792 32.83915, -117.27775 32...",3204
5,Ventura,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",53


In [59]:
#subset for total  service vehicles by county
county_passengers = aggregation_sum(subset2, 'COUNTY_NAME')
county_passengers

,COUNTY_NAME,geometry,upt_total_2019
0,Los Angeles,"MULTIPOINT (-118.84339 34.03152, -118.82113 34...","379,718,121379,718,121379,718,121379,718,12137..."
1,Orange,"MULTIPOINT (-118.10607 33.74889, -118.10577 33...","820,829820,829820,829820,829820,829820,829820,..."
2,Riverside,"MULTIPOINT (-117.55461 33.99433, -117.46784 33...","10,863,53040,743,654"
3,San Bernardino,"MULTIPOINT (-117.73260 34.00136, -117.72986 34...","10,863,53010,863,53010,863,53010,863,53010,863..."
4,San Diego,"MULTIPOINT (-117.27792 32.83915, -117.27775 32...","85,357,49585,357,49585,357,49585,357,49585,357..."
5,Ventura,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...","379,718,121379,718,121379,718,121379,718,12137..."


In [18]:
#combine the 2 metrics together
county = gpd.sjoin(county_vehicles, county_stops, how='inner')

In [19]:
county = county.drop(columns = ['COUNTY_NAME_right','index_right'])

In [20]:
county = county.rename(columns = {'COUNTY_NAME_left': 'COUNTY_NAME', 'stop_id':'count_of_unique_stops','servvehicles_2019':'total_service_vehicles'})

In [21]:
county

,COUNTY_NAME,geometry,stop_id_left,stop_id_right
0,Los Angeles,"MULTIPOINT (-118.84339 34.03152, -118.82790 34...",16900,16900
1,Orange,"MULTIPOINT (-118.10607 33.74889, -118.10577 33...",5579,5579
2,Riverside,"MULTIPOINT (-117.55461 33.99433, -117.46784 33...",5,5
3,San Bernardino,"MULTIPOINT (-117.73260 34.00136, -117.73217 33...",2331,2331
4,San Diego,"MULTIPOINT (-117.27792 32.83915, -117.27792 32...",4569,4569
5,Ventura,"MULTIPOINT (-118.88283 34.18006, -118.88283 34...",55,55


In [22]:
#merge again for polygon
county_geo  = pd.merge(county.drop(columns="geometry"), geojson,  on="COUNTY_NAME")

In [23]:
#filter out the islands
county_geo = county_geo.loc[county_geo['ISLAND'] != 'Channel Islands'] 

In [24]:
#add centroid for each county 
county_geo['centroid'] = county_geo.geometry.centroid

/tmp/ipykernel_1342/354039997.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



In [47]:
county_geo.head(5)

,COUNTY_NAME,stop_id_left,stop_id_right,OBJECTID,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry,centroid
0,Los Angeles,16900,16900,19,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.178693,1.002853,"MULTIPOLYGON (((-117.66733 34.79317, -117.6672...",POINT (-118.21689 34.36117)
3,Orange,5579,5579,30,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.158610,0.201290,"MULTIPOLYGON (((-118.11526 33.74164, -118.1150...",POINT (-117.76121 33.70309)
4,Riverside,5,5,33,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.952959,1.839965,"MULTIPOLYGON (((-114.43559 34.07847, -114.4356...",POINT (-115.99382 33.74374)
5,San Bernardino,2331,2331,36,SBD,36,36,071,None,{E4DF6870-0D0B-40C1-AD3B-A60A72792CFD},10.455614,5.132310,"MULTIPOLYGON (((-115.41359 35.62499, -115.3970...",POINT (-116.17852 34.84146)
6,San Diego,4569,4569,37,SDG,37,37,073,None,{414826EC-689E-4084-BD03-5195DF2748BF},4.725270,1.064552,"MULTIPOLYGON (((-117.24152 33.44879, -117.2413...",POINT (-116.73509 33.03422)


In [26]:
test = county_geo.drop(columns = ['centroid'])

### Visualizations
* Make one chart for comparing trips per stop by operators, and another chart for comparing it by counties. Use a function to do this.
* Make 1 map for comparing trips per stop by counties. Use `shared_utils.map_utils` to do this.
* Visualizations should follow the Cal-ITP style guide.
* More on `folium` and `ipyleaflet`: https://github.com/jorisvandenbossche/geopandas-tutorial/blob/master/05-more-on-visualization.ipynb

#### Bar chart
* Make it into a function.
* Don't print out bar chart b/c this causes a file saving error

In [27]:
def bar_chart(df, x_col, y_col):
    y_col_stripped = y_col.replace('_',' ')
    chart_title = f"{y_col_stripped}" 
    chart = (alt.Chart(df).mark_bar().encode(
                 x=alt.X(x_col, title=x_col),
                 y=alt.Y(y_col, title=y_col_stripped),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.FIVETHIRTYEIGHT_CATEGORY_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250,
                 title = chart_title)
            )
    #return chart
    display(chart)

In [36]:
#bar_chart(county, 'COUNTY_NAME', 'stop_id_left') 

In [37]:
#bar_chart(county, 'COUNTY_NAME', 'stop_id_left') 

#### Map - HELP
* Object of type Point is not JSON serializable

In [30]:
choropleth_dict = ({
            "layer_name": 'my layer',
            "MIN_VALUE": int(0),
            "MAX_VALUE": int(100),
            "plot_col_name": 'County_Name',
            "fig_width": '100%',
            "fig_height": '100%',
            "fig_min_width_px": '600px',
            "fig_min_height_px": '600px'})

In [31]:
colorscale = branca.colormap.StepColormap(
                colors=["gray", "green", "navy"], 
                #index=[2_000, 4_000, 8_000],
                #vmin=0, vmax=15_000,
)


In [32]:
def grab_region_centroids():
    # This parquet is created in shared_utils/shared_data.py
    df = pd.read_parquet(
        "gs://calitp-analytics-data/data-analyses/ca_county_centroids.parquet")
    
    df = df.assign(
        centroid = df.centroid.apply(lambda x: x.tolist())
    )    
    
    # Manipulate parquet file to be dictionary to use in map_utils
    region_centroids = dict(
        zip(df.county_name, 
            df[["centroid", "zoom"]].to_dict(orient="records")
           )
    )

    return region_centroids

In [33]:
REGION_CENTROIDS = grab_region_centroids()

In [34]:
#shared_utils.map_utils.make_ipyleaflet_choropleth_map(test, 
                                                     # plot_col = 'stop_id_left',
                                                     # geometry_col = 'COUNTY_NAME', 
                                                     # choropleth_dict = choropleth_dict, 
                                                      #colorscale = colorscale,
                                                     # zoom=REGION_CENTROIDS["CA"]["zoom"], centroid = REGION_CENTROIDS["CA"]["centroid"])